# Machine Learning in Python - Project 

Due Friday, Apr 10th by 4 pm.

*Include contributors names here*

## Setup

*Install any packages here, define any functions if neeed, and load data*

In [1]:
# Add any additional libraries or submodules below

# Data libraries
import pandas as pd
import numpy as np

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn modules
import sklearn

In [2]:
# Load data  
df = pd.read_csv("unicef_malawi.csv")
df.head()
df['FCF26_clean'] = df['FCF26'].replace('NO RESPONSE', np.nan)
print(df["FCF26_clean"].isna().sum())

126


In [3]:
data_selected = pd.read_csv("model_data_selected.csv")
transformation = pd.read_csv("transformation_log.csv")

In [4]:
missing_summary = data_selected.isna().mean().sort_values(ascending=False)
print(missing_summary)

MA2                 0.239718
CL13                0.150399
FCD2A               0.141114
FCD2D               0.138198
FCD2E               0.138198
FCD2H               0.138122
WS14                0.074279
LS4                 0.070442
VT1                 0.024171
VT22B               0.024095
VT22A               0.023097
VT21                0.023020
LS1                 0.022867
VT20                0.022713
LS3                 0.022560
VT22X               0.022406
AF10                0.022023
WB6A                0.021486
WB4                 0.021409
MA3                 0.014886
WS11                0.000153
CL2                 0.000000
ethnicity           0.000000
HH7                 0.000000
MSTATUS             0.000000
CSURV               0.000000
HC5                 0.000000
WS7                 0.000000
HC4                 0.000000
WS1                 0.000000
child_depression    0.000000
dtype: float64


In [5]:
# 处理CL13
# 先复制一份数据，以免修改原数据
df_model = data_selected.copy()

# Step 1: CL12 = NO → CL13 = 0（只填缺失）
df_model["CL13"] = pd.to_numeric(df_model["CL13"], errors="coerce")
mask_no = (df["CL12"] == "FALSE") & (df_model["CL13"].isna())
df_model.loc[mask_no, "CL13"] = 0

# Step 2: 计算 median（只在 CL12 == YES 中）
median_yes = df_model.loc[
    (df["CL12"] == "TRUE") & (df_model["CL13"].notna()),
    "CL13"
].median()

# Step 3: CL12 = YES → 用 median 填补
mask_yes = (df["CL12"] == "TRUE") & (df_model["CL13"].isna())
df_model.loc[mask_yes, "CL13"] = median_yes

# 检查
print(df_model["CL13"].isna().sum())

1962


In [26]:
#填补FCD2H-FCD2A
fcd_vars = [
    "FCD2H","FCD2E","FCD2D","FCD2A"
]

for col in fcd_vars:
    mode_val = df_model[col].mode()[0]
    df_model[col] = df_model[col].fillna(mode_val)

#填补WB4
# 计算中位数
median_wb4 = df_model["WB4"].median()
# 填补
df_model["WB4"] = df_model["WB4"].fillna(median_wb4)

#填补WB6A
mode_wb6a = df_model["WB6A"].mode()[0]
df_model["WB6A"] = df_model["WB6A"].fillna(mode_wb6a)

#填补VT22A和VT22B
vt_vars = ["VT22A", "VT22B", "VT1", "VT22X"]
for col in vt_vars:
    mode_val = df_model[col].mode()[0]
    df_model[col] = df_model[col].fillna(mode_val)

#填补VT20和VT21
multi_cat_vars = ["VT20", "VT21"]
for col in multi_cat_vars:
    mode_val = df_model[col].mode()[0]
    df_model[col] = df_model[col].fillna(mode_val)

#填补MA2
# 计算中位数
median_ma2 = df_model["MA2"].median()
# 填补
df_model["MA2"] = df_model["MA2"].fillna(median_ma2)

#填补MA3
mode_ma3 = df_model["MA3"].mode()[0]
df_model["MA3"] = df_model["MA3"].fillna(mode_ma3)

#填补AF10
mode_af10 = df_model["AF10"].mode()[0]
df_model["AF10"] = df_model["AF10"].fillna(mode_af10)

#填补LS1
mode_ls1 = df_model["LS1"].mode()[0]
df_model["LS1"] = df_model["LS1"].fillna(mode_ls1)

#填补LS3
mode_ls3 = df_model["LS3"].mode()[0]
df_model["LS3"] = df_model["LS3"].fillna(mode_ls3)

#填补LS4
mode_ls4 = df_model["LS4"].mode()[0]
df_model["LS4"] = df_model["LS4"].fillna(mode_ls4)

#填补WS11
mode_ws11 = df_model["WS11"].mode()[0]
df_model["WS11"] = df_model["WS11"].fillna(mode_ws11)

#填补WS14
mode_ws14 = df_model["WS14"].mode()[0]
df_model["WS14"] = df_model["WS14"].fillna(mode_ws14)

In [27]:
missing_summary = df_model.isna().mean().sort_values(ascending=False)
print(missing_summary)

CL13                0.150552
ethnicity           0.000000
HH7                 0.000000
CL2                 0.000000
FCD2H               0.000000
FCD2E               0.000000
FCD2D               0.000000
FCD2A               0.000000
WB4                 0.000000
WB6A                0.000000
VT22A               0.000000
VT22B               0.000000
VT20                0.000000
VT21                0.000000
VT1                 0.000000
VT22X               0.000000
MA2                 0.000000
MA3                 0.000000
MSTATUS             0.000000
AF10                0.000000
LS1                 0.000000
LS3                 0.000000
LS4                 0.000000
CSURV               0.000000
HC5                 0.000000
HC4                 0.000000
WS7                 0.000000
WS1                 0.000000
WS11                0.000000
WS14                0.000000
child_depression    0.000000
dtype: float64


# Introduction

*This section should include a brief introduction to the task and the data (assume this is a report you are delivering to a professional body (e.g. UNICEF)).*

*Briefly outline the approaches being used and the conclusions that you are able to draw.*

# Exploratory Data Analysis and Feature Engineering

*Include a detailed discussion of the data with a particular emphasis on the features of the data that are relevant for the subsequent modeling. Including visualizations of the data is strongly encouraged - all code and plots must also be described in the write up. Think carefully about whether each plot needs to be included in your final draft and the appropriate type of plot and summary for each variable type - your report should include figures but they should be as focused and impactful as possible.*

*You should also split your data into training and testing sets, ideally before you look to much into the features and relationships with the target.*

*Additionally, this section should also motivate and describe any preprocessing / feature engineering of the data. Pipelines should be used and feature engineering steps that are be performed as part of an sklearn pipeline can be mentioned here but should be implemented in the following section.*

*All code and figures should be accompanied by text that provides an overview / context to what is being done or presented.*

# Model Fitting and Tuning

*In this section you should detail and motivate your choice of model and describe the process used to refine, tune, and fit that model. You are encouraged to explore different models but you should NOT include a detailed narrative or code of all of these attempts. At most this section should very briefly mention the methods explored and why they were rejected - most of your effort should go into describing the final model you are using and your process for tuning and validating it.*

*This section should include the full implementation of your final model, including all necessary validation. As with figures, any included code must also be addressed in the text of the document.*

*You should also include a baseline model of your choice and provide a comparison of your model with the baseline model on the test data. You should briefly describe the baseline model considered.*

# Interpretation, Discussion & Conclusions

*In this section you should provide a general overview of your final model, its performance, and reliability. You should discuss what the implications of your model are in terms of the included features, estimated parameters and relationships, predictive performance, and anything else you think is relevant.*

*This should be written with a target audience of a government official, who understands the issues associated with mental health but may only have university level mathematics (not necessarily postgraduate statistics or machine learning). Your goal should be to highlight to this audience how your model can useful. You should also discuss potential limitations or directions of future improvement of your model.*

*Finally, you should include recommendations on factors that may increase the risk of depression, which may be useful for the government officials and health care workers to improve their understanding of the condition, and potentially assit in the development of effective social and health policies and interventions.*

*Keep in mind that a negative result, i.e. a model that does not work well predictively, that is well explained and justified in terms of why it failed will likely receive higher marks than a model with strong predictive performance but with poor or incorrect explanations / justifications.*

# Generative AI statement

*Include a statement on how generative AI was used in the project and report.*

# References

*Include references if any*

In [ ]:
# Run the following to render to PDF
!jupyter nbconvert --to pdf project.ipynb